NOTE: This is the notebook used for calculating inference on your given datasets. Since the inference doesn't require lot of computation, you can load any default GPU runtime in jupyter notebook and run it directly without using your colab credits. It can be used in any free colab GPU.

#### **What is Word Error Rate (WER)**

**Word Error Rate (WER)** is a common metric used to evaluate the performance of automatic speech recognition (ASR) systems. It measures the percentage of words that were incorrectly predicted by the ASR system compared to a reference transcription. WER is calculated using the following formula:


#### *WER = Substitutions + Deletions + Insertions/Total Words in Reference*



- **Substitutions**: The number of words that were incorrectly transcribed as different words.
- **Deletions**: The number of words that were omitted from the transcription.
- **Insertions**: The number of extra words that were added to the transcription.

A lower WER indicates better ASR performance. For instance, a WER of 5% means that 5 out of every 100 words were transcribed incorrectly.

## Example Calculation:

```python
from jiwer import wer

# Reference transcriptions (ground truth)
references = ["the quick brown fox jumps over the lazy dog"]

# Hypothesis transcriptions (ASR output)
hypotheses = ["the quick brown fox jump over the lazy dogs"]

# Calculate WER
error_rate = wer(references, hypotheses)
error_rate_percentage = error_rate * 100
print(f"Word Error Rate (WER): {error_rate_percentage:.2f}%")


**Installation and Setup Instructions**

This section covers the installation of various Python libraries and packages necessary for our ASR project using the Whisper model.

1. **datasets**: This library is used for handling and processing datasets. It provides easy access to a wide range of datasets and tools for processing them.

2. **transformers**: This library from Hugging Face allows us to use state-of-the-art models like Whisper for various NLP tasks, including ASR.

3. **librosa**: This library is used for audio and music processing. It helps in loading, analyzing, and extracting features from audio files.

4. **evaluate**: This library provides various evaluation metrics, which are essential for assessing the performance of our ASR model.

5. **jiwer**: This library is used for calculating Word Error Rate (WER), a common metric to evaluate the accuracy of ASR systems by comparing the predicted transcriptions to the reference transcriptions.

6. **ipython-autotime**: This extension helps measure the execution time of each code cell in Jupyter notebooks, which is useful for performance evaluation.

In [ ]:
!pip install datasets>=2.6.1
!pip install git+https://github.com/huggingface/transformers
!pip install librosa
!pip install evaluate>=0.30
!pip install jiwer
!pip install ipython-autotime
!pip install accelerate -U
%load_ext autotime
!pip install -U datasets[audio]

In [ ]:
import torch
import transformers
import datasets
import pyarrow
from importlib.metadata import version

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Jiwer:", version("jiwer"))
print("PyArrow:", pyarrow.__version__)

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

This block of code checks if the GPU information contains the word 'failed'. If it does, it indicates that no GPU is connected. Otherwise, it prints the GPU information.

In [ ]:
from transformers import pipeline
from transformers.models.whisper.english_normalizer import BasicTextNormalizer
from datasets import load_dataset, Audio
import evaluate
from datasets import load_dataset, DatasetDict

This section covers the imports of various Python libraries and modules necessary for our ASR project using the Whisper

In [ ]:
from huggingface_hub import notebook_login

notebook_login()
# Login following the prompted instructions.

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

# Load the model and processor
#Example for Model ID: model_id = "huggingface_repo/model_path"
#model_id = "openai/whisper-tiny"   #Provide the path to the model you want to use for inference. We will use whisper-tiny (the smallest one) for this notebook. You can see the family of whisper models at: https://github.com/openai/whisper
model_id = "AVryon/whisper-tiny_to_korean_accent_29oct"
model = WhisperForConditionalGeneration.from_pretrained(model_id)
processor = WhisperProcessor.from_pretrained(model_id)

In [ ]:
# This block is used to resolve the transformers conflicts between versions. Whisper was trained with an old transformer version, which might have staled arguments the its config.
gen_params = model.config._get_generation_parameters()
for key, value in gen_params.items():
    setattr(model.generation_config, key, value)  # keep the setting where generate() expects it
    setattr(model.config, key, None)              # clear it from the model config

The model ID, "openai/Whisper-tiny", specifies the path to the pre-trained Whisper model. This ID is used to load the model and processor. You can change this path to your own trained model to do inference with your own finetuned model. Example model_id = "huggingface_repo/finetuned_model"

In [ ]:
# Move the model to GPU if available
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

This line moves the Whisper model to the specified device (either GPU or CPU) to ensure that computations are performed on the appropriate hardware.

In [ ]:
from datasets import load_dataset

# Load the dataset
references = load_dataset("naharte/korean_english", split="test")
print(references)
print("Columns:", references.column_names)
print("Number of recordings:", len(references))
print("First reference:", references[0]["transcription"])
print("Audio sampling rate:", references[0]["audio"]["sampling_rate"])

This block loads the dataset specified by the identifier "naharte/spanish_english" and selects the test split. The loaded dataset is stored in the references variable. If you want to use both the test and train splits for inference, you can pass "test+train" in the split parameter

In [ ]:
def transcribe_audio(batch):
    audio = batch["audio"]["array"]  # Ensure you access the raw audio array
    inputs = processor(audio, return_tensors="pt", sampling_rate=16000)
    input_features = inputs.input_features.to(model.device)

    with torch.no_grad():
        predicted_ids = model.generate(input_features) # these are just tensor numbers, not the actual characters and words yet.

    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True) # this step converts numbers to text.
    return transcription[0]

This function processes the audio data, generates predictions using the Whisper model, and returns the transcribed text.

In [ ]:
hypotheses = references.map(lambda batch: {"transcription": transcribe_audio(batch)}, remove_columns=["audio"]) # This code might take 2-3 minutes to run.

This block uses the map method from the datasets library to apply the transcribe_audio function to each batch in the references dataset. The lambda function ensures that the transcribe_audio function is applied to each batch, and the resulting transcription is stored in the transcription column. The remove_columns parameter specifies that the audio column should be removed from the resulting dataset.

In [ ]:
print("Reference: ", references['transcription'][5])
print("Hypothesis: ", hypotheses['transcription'][5])

# for i in range(len(references)):
#     print(f"\n--- Recording {i+1} ---")
#     print("Reference: ", references["transcription"][i])
#     print("Hypothesis:", hypotheses["transcription"][i])


In [ ]:
from jiwer import wer

# Calculate WER
ref_text = list(references["transcription"])
hyp_text = list(hypotheses["transcription"])

error_rate = wer(ref_text, hyp_text)
error_rate_percentage = error_rate * 100
print(f"Word Error Rate (WER): {error_rate_percentage:.2f}%")


In [ ]:

from jiwer import wer
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

normalizer = BasicTextNormalizer()

ref_text = list(references["transcription"])
hyp_text = list(hypotheses["transcription"])

raw_wer = wer(ref_text, hyp_text)

normalized_refs = [normalizer(t) for t in ref_text]
normalized_hyps = [normalizer(t) for t in hyp_text]

normalized_wer = wer(normalized_refs, normalized_hyps)

print(f"Raw WER: {raw_wer * 100:.2f}%")
print(f"Normalized WER: {normalized_wer * 100:.2f}%")


This code snippet calculates and displays the Word Error Rate between the reference and hypothesis transcriptions for your given accented dataset

In [ ]:

import pandas as pd
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

results_dir = Path(
    "/content/assignment_results"
)
results_dir.mkdir(parents=True, exist_ok=True)

results_df = pd.DataFrame({
    "reference": ref_text,
    "hypothesis": hyp_text,
    "normalized_reference": normalized_refs,
    "normalized_hypothesis": normalized_hyps
})

output_path = results_dir / "korean_finetuned_decoding_predictions.csv"
results_df.to_csv(output_path, index=False)

print("Saved to:", output_path)
print("Number of rows:", len(results_df))


In [ ]:
import pandas as pd

expected_counts = {
    "canadian": 407,
    "indian": 650,
    "chinese": 629,
    "japanese": 700,
    "korean": 450
}

for accent, expected in expected_counts.items():
    path = results_dir / f"{accent}_baseline_predictions.csv"
    df = pd.read_csv(path, keep_default_na=False)

    assert len(df) == expected, f"{accent}: incorrect row count"
    assert df["normalized_reference"].str.strip().ne("").all(), (
        f"{accent}: empty normalized reference"
    )

    score = wer(
        df["normalized_reference"].tolist(),
        df["normalized_hypothesis"].tolist()
    )

    print(f"{accent.capitalize():10s} | "
          f"{len(df):4d} samples | "
          f"Normalized WER: {score * 100:.2f}%")
